<a href="https://colab.research.google.com/github/luismiguelaristi/MecanismosPythonUPB/blob/main/ManipuladorParallelo_2GDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis cinemático de Manipulador Paralelo Plano 2-GDL

![manipulador_paralelo.png](https://via.placeholder.com/400x300?text=Manipulador+Paralelo+Plano)

## Descripción
Un manipulador paralelo plano de 2 grados de libertad (GDL) es un mecanismo conformado por una base fija, dos cadenas cinemáticas idénticas, y un efector final que se mueve en el plano. Este tipo de manipuladores ofrecen ventajas como alta rigidez, carga útil elevada y velocidad de movimiento. A diferencia de los manipuladores seriales, los paralelos reparten la carga entre todos sus eslabones, lo que permite diseños más compactos y precisos.

El mecanismo analizado en este cuaderno contiene:
- **2 actuadores** en la base (dos pares revolucionarios)
- **2 eslabones motores** (brazos)
- **1 acoplador** que conecta ambos brazos
- **Pares revoluta**: 5 en total
  1. Par revoluta en base izquierda
  2. Par revoluta entre brazo izquierdo y acoplador
  3. Par revoluta en base derecha
  4. Par revoluta entre brazo derecho y acoplador
  5. Par revoluta en el efector final (punto de control)

## Diagrama Cinemático

```
Efector Final P(x,y)
      │
   ┌──┴──┐
   │Acopl│ Long. L3
   └──┬──┘
     ╱ ╲
    ╱   ╲
 L2╱     ╲L4
  ╱       ╲
 ╱         ╲
O1━━━base━━━O2
θ1          θ2
```

## Variables y Parámetros

### Parámetros del Mecanismo (Longitudes de eslabones)
- $L_1$: Longitud del brazo izquierdo
- $L_2$: Longitud del brazo derecho  
- $L_3$: Longitud del acoplador (semi)
- $d$: Distancia entre centros de rotación en la base
- Se consideran dimensiones simétricas para simplificar

### Variables de Entrada (Ángulos de los Actuadores)
- $\theta_1$: Ángulo del brazo izquierdo
- $\theta_2$: Ángulo del brazo derecho

### Variables de Salida
- $x_P, y_P$: Posición del efector final
- $\phi$: Orientación del acoplador (si aplica)

## Ecuaciones Vectoriales

### Cinemática Directa (Forward Kinematics)

La posición del efector final se obtiene a partir de los ángulos de entrada $\theta_1$ y $\theta_2$:

**Cadena Left:**
- $A_1 = (d/2, 0) + L_1(\cos\theta_1, \sin\theta_1)$

**Cadena Right:**
- $A_2 = (-d/2, 0) + L_2(\cos\theta_2, \sin\theta_2)$

**Efector Final (punto medio del acoplador):**

$$P_x = \frac{A_{1x} + A_{2x}}{2}$$
$$P_y = \frac{A_{1y} + A_{2y}}{2}$$

### Cinemática Inversa (Inverse Kinematics)

Dado una posición deseada del efector final $(x_P, y_P)$, se deben encontrar los ángulos $\theta_1$ y $\theta_2$:

Las distancias desde la base a los puntos del acoplador deben satisfacer:

$$|A_1 - O_1| = L_1 \quad \text{y} \quad |A_2 - O_2| = L_2$$

$$|A_1 - A_2| \approx L_3$$

Resolviendo estas ecuaciones se obtienen los valores de $\theta_1$ y $\theta_2$.

## Implementación Numérica

### Importar módulos

In [ ]:
%reset -sf

import numpy as np
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle
import pandas as pd

### Parámetros del Mecanismo (Modificables)

Modifique los siguientes parámetros para cambiar las dimensiones del manipulador paralelo:

In [ ]:
# =====================================================
# PARÁMETROS MODIFICABLES DEL MECANISMO
# =====================================================

# Longitudes de los eslabones [mm]
L1 = 100.0  # Longitud del brazo izquierdo
L2 = 100.0  # Longitud del brazo derecho
L3 = 80.0   # Longitud del acoplador

# Distancia entre pares revolucionarios en la base [mm]
d = 120.0   # Separación entre base izquierda y derecha

# Condiciones iniciales para análisis
theta1_ini = np.radians(30)   # Ángulo inicial del brazo izquierdo [rad]
theta2_ini = np.radians(150)  # Ángulo inicial del brazo derecho [rad]

# Información del mecanismo
print("="*50)
print("PARÁMETROS DEL MANIPULADOR PARALELO PLANO 2-GDL")
print("="*50)
print(f"Longitud brazo izquierdo (L1):     {L1} mm")
print(f"Longitud brazo derecho (L2):       {L2} mm")
print(f"Longitud acoplador (L3):           {L3} mm")
print(f"Separación entre actuadores (d):  {d} mm")
print(f"Ángulo inicial brazo izq. (θ1):   {np.degrees(theta1_ini):.2f}°")
print(f"Ángulo inicial brazo der. (θ2):   {np.degrees(theta2_ini):.2f}°")
print("="*50)

### Cinemática Directa (Forward Kinematics)

In [ ]:
def cinemática_directa(theta1, theta2):
    """
    Calcula la posición del efector final dado los ángulos de entrada.
    
    Parámetros:
    -----------
    theta1 : float
        Ángulo del brazo izquierdo [rad]
    theta2 : float
        Ángulo del brazo derecho [rad]
    
    Retorna:
    --------
    tuple : (x_p, y_p) - Posición del efector final
    """
    
    # Posiciones de los puntos de acoplamiento (extremos de los brazos)
    # Brazo izquierdo
    A1_x = -d/2 + L1 * np.cos(theta1)
    A1_y = L1 * np.sin(theta1)
    
    # Brazo derecho
    A2_x = d/2 + L2 * np.cos(theta2)
    A2_y = L2 * np.sin(theta2)
    
    # Efector final: punto medio del segmento A1-A2
    x_p = (A1_x + A2_x) / 2
    y_p = (A1_y + A2_y) / 2
    
    return x_p, y_p, A1_x, A1_y, A2_x, A2_y

# Prueba con las condiciones iniciales
x_p, y_p, A1_x, A1_y, A2_x, A2_y = cinemática_directa(theta1_ini, theta2_ini)

print(f"\nPosición del Efector Final:")
print(f"x_p = {x_p:.2f} mm, y_p = {y_p:.2f} mm")
print(f"\nPuntos de acoplamiento:")
print(f"A1 = ({A1_x:.2f}, {A1_y:.2f}) mm")
print(f"A2 = ({A2_x:.2f}, {A2_y:.2f}) mm")
print(f"Distancia A1-A2 = {np.sqrt((A2_x-A1_x)**2 + (A2_y-A1_y)**2):.2f} mm")

### Análisis del Espacio de Trabajo (Workspace Analysis)

In [ ]:
def estimar_espacio_trabajo(rango_theta1, rango_theta2, num_puntos=50):
    """
    Estima el espacio de trabajo barriendo los ángulos de entrada.
    
    Parámetros:
    -----------
    rango_theta1 : tuple
        (min, max) ángulo para brazo izquierdo [rad]
    rango_theta2 : tuple
        (min, max) ángulo para brazo derecho [rad]
    num_puntos : int
        Número de puntos en cada dirección
    
    Retorna:
    --------
    dict : Coordenadas del espacio de trabajo
    """
    
    theta1_range = np.linspace(rango_theta1[0], rango_theta1[1], num_puntos)
    theta2_range = np.linspace(rango_theta2[0], rango_theta2[1], num_puntos)
    
    workspace_x = []
    workspace_y = []
    
    for theta1 in theta1_range:
        for theta2 in theta2_range:
            x_p, y_p, _, _, _, _ = cinemática_directa(theta1, theta2)
            workspace_x.append(x_p)
            workspace_y.append(y_p)
    
    return {'x': np.array(workspace_x), 'y': np.array(workspace_y)}

# Calcular el espacio de trabajo
rango_theta1 = (np.radians(0), np.radians(180))
rango_theta2 = (np.radians(0), np.radians(180))

workspace = estimar_espacio_trabajo(rango_theta1, rango_theta2, num_puntos=40)

print(f"Espacio de trabajo calculado: {len(workspace['x'])} puntos")
print(f"Rango en X: [{workspace['x'].min():.2f}, {workspace['x'].max():.2f}] mm")
print(f"Rango en Y: [{workspace['y'].min():.2f}, {workspace['y'].max():.2f}] mm")

### Visualización del Espacio de Trabajo

In [ ]:
# Visualizar espacio de trabajo
plt.figure(figsize=(10, 8))
plt.scatter(workspace['x'], workspace['y'], c='blue', s=20, alpha=0.6, label='Espacio de trabajo')

# Marcar posición inicial del efector final
x_ini, y_ini, _, _, _, _ = cinemática_directa(theta1_ini, theta2_ini)
plt.plot(x_ini, y_ini, 'r*', markersize=15, label='Posición inicial')

# Marcar las bases de los actuadores
plt.plot([-d/2, d/2], [0, 0], 'ks', markersize=10, label='Actuadores')

plt.xlabel('x [mm]', fontsize=12)
plt.ylabel('y [mm]', fontsize=12)
plt.title('Espacio de Trabajo del Manipulador Paralelo Plano 2-GDL', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.legend()
plt.tight_layout()
plt.show()

### Consolidación de Datos con Pandas

In [ ]:
def generar_data_cinemática(rango_theta1, num_posiciones=30):
    """
    Genera datos cinemáticos para un rango de movimiento del primer actuador.
    El segundo actuador se mantiene en una posición fija.
    
    Parámetros:
    -----------
    rango_theta1 : tuple
        (min, max) ángulo para brazo izquierdo [rad]
    num_posiciones : int
        Número de posiciones a analizar
    
    Retorna:
    --------
    pd.DataFrame : Datos cinemáticos
    """
    
    theta1_array = np.linspace(rango_theta1[0], rango_theta1[1], num_posiciones)
    theta2_fixed = np.radians(90)  # Fijo en 90°
    
    data = {
        'theta1_deg': np.degrees(theta1_array),
        'theta2_deg': np.full(num_posiciones, np.degrees(theta2_fixed)),
        'x_p': [],
        'y_p': [],
        'A1_x': [],
        'A1_y': [],
        'A2_x': [],
        'A2_y': [],
        'distancia_A1A2': []
    }
    
    for theta1 in theta1_array:
        x_p, y_p, A1_x, A1_y, A2_x, A2_y = cinemática_directa(theta1, theta2_fixed)
        dist = np.sqrt((A2_x - A1_x)**2 + (A2_y - A1_y)**2)
        
        data['x_p'].append(x_p)
        data['y_p'].append(y_p)
        data['A1_x'].append(A1_x)
        data['A1_y'].append(A1_y)
        data['A2_x'].append(A2_x)
        data['A2_y'].append(A2_y)
        data['distancia_A1A2'].append(dist)
    
    return pd.DataFrame(data)

# Generar datos
df = generar_data_cinemática((np.radians(0), np.radians(180)), num_posiciones=25)

print("\nDatos Cinemáticos (primeras 10 filas):")
print(df.head(10))

### Gráficas de Análisis Cinemático

In [ ]:
# Crear figura con subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Gráfica 1: X vs Theta1
axes[0, 0].plot(df['theta1_deg'], df['x_p'], 'b-o', linewidth=2, markersize=4)
axes[0, 0].set_xlabel('Ángulo del brazo izquierdo θ₁ [°]', fontsize=10)
axes[0, 0].set_ylabel('Posición x_p [mm]', fontsize=10)
axes[0, 0].set_title('Posición X del Efector Final')
axes[0, 0].grid(True, alpha=0.3)

# Gráfica 2: Y vs Theta1
axes[0, 1].plot(df['theta1_deg'], df['y_p'], 'g-o', linewidth=2, markersize=4)
axes[0, 1].set_xlabel('Ángulo del brazo izquierdo θ₁ [°]', fontsize=10)
axes[0, 1].set_ylabel('Posición y_p [mm]', fontsize=10)
axes[0, 1].set_title('Posición Y del Efector Final')
axes[0, 1].grid(True, alpha=0.3)

# Gráfica 3: Distancia A1-A2 vs Theta1
axes[1, 0].plot(df['theta1_deg'], df['distancia_A1A2'], 'r-o', linewidth=2, markersize=4)
axes[1, 0].axhline(y=L3, color='k', linestyle='--', label=f'L₃ = {L3} mm')
axes[1, 0].set_xlabel('Ángulo del brazo izquierdo θ₁ [°]', fontsize=10)
axes[1, 0].set_ylabel('Distancia A1-A2 [mm]', fontsize=10)
axes[1, 0].set_title('Distancia entre Puntos de Acoplamiento')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Gráfica 4: Trayectoria del efector final
axes[1, 1].plot(df['x_p'], df['y_p'], 'b-o', linewidth=2, markersize=6)
axes[1, 1].plot(df['x_p'][0], df['y_p'][0], 'g*', markersize=15, label='Inicio')
axes[1, 1].plot(df['x_p'][-1], df['y_p'][-1], 'r*', markersize=15, label='Final')
axes[1, 1].set_xlabel('x_p [mm]', fontsize=10)
axes[1, 1].set_ylabel('y_p [mm]', fontsize=10)
axes[1, 1].set_title('Trayectoria del Efector Final')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].axis('equal')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### Animación del Mecanismo

Función para visualizar el movimiento del manipulador paralelo

In [ ]:
def visualizar_configuraciones(theta1_range, num_configs=9):
    """
    Visualiza varias configuraciones del mecanismo.
    
    Parámetros:
    -----------
    theta1_range : tuple
        (min, max) ángulo para visualizar [rad]
    num_configs : int
        Número de configuraciones a mostrar
    """
    
    theta1_values = np.linspace(theta1_range[0], theta1_range[1], num_configs)
    theta2_fixed = np.radians(90)
    
    fig, axes = plt.subplots(3, 3, figsize=(14, 12))
    axes = axes.flatten()
    
    # Calcular espacios de trabajo para contexto
    ws = estimar_espacio_trabajo(theta1_range, (theta2_fixed-np.radians(20), theta2_fixed+np.radians(20)), num_puntos=30)
    
    for idx, theta1 in enumerate(theta1_values):
        ax = axes[idx]
        
        # Dibujar espacio de trabajo (contexto)
        ax.scatter(ws['x'], ws['y'], c='lightblue', s=10, alpha=0.5, zorder=1)
        
        # Calcular posiciones
        x_p, y_p, A1_x, A1_y, A2_x, A2_y = cinemática_directa(theta1, theta2_fixed)
        
        # Dibujar eslabones
        # Brazo izquierdo
        ax.plot([-d/2, A1_x], [0, A1_y], 'b-', linewidth=3, label='Brazo izq.')
        # Brazo derecho
        ax.plot([d/2, A2_x], [0, A2_y], 'r-', linewidth=3, label='Brazo der.')
        # Acoplador
        ax.plot([A1_x, A2_x], [A1_y, A2_y], 'g-', linewidth=3, label='Acoplador')
        
        # Dibujar pares revolucionarios
        ax.plot([-d/2, d/2], [0, 0], 'ks', markersize=8, label='Base')
        ax.plot([A1_x, A2_x], [A1_y, A2_y], 'go', markersize=8, zorder=3)
        ax.plot(x_p, y_p, 'r*', markersize=15, label='Efector final', zorder=4)
        
        # Configurar ejes
        ax.set_xlim(-150, 150)
        ax.set_ylim(-50, 200)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
        ax.set_xlabel('x [mm]', fontsize=9)
        ax.set_ylabel('y [mm]', fontsize=9)
        ax.set_title(f'θ₁ = {np.degrees(theta1):.1f}°, θ₂ = {np.degrees(theta2_fixed):.1f}°', fontsize=10)
        
        if idx == 0:
            ax.legend(fontsize=8, loc='upper left')
    
    # Ocultar ejes sobrantes
    for idx in range(len(theta1_values), 9):
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# Visualizar diferentes configuraciones
visualizar_configuraciones((np.radians(0), np.radians(180)), num_configs=9)

## Análisis de Velocidades

### Matriz Jacobiana

La relación entre velocidades articulares y velocidades cartesianas está dada por:

$$\begin{pmatrix} \dot{x} \\ \dot{y} \end{pmatrix} = \mathbf{J} \begin{pmatrix} \dot{\theta}_1 \\ \dot{\theta}_2 \end{pmatrix}$$

donde la matriz Jacobiana es:

$$\mathbf{J} = \begin{pmatrix} \frac{\partial x_p}{\partial \theta_1} & \frac{\partial x_p}{\partial \theta_2} \\ \frac{\partial y_p}{\partial \theta_1} & \frac{\partial y_p}{\partial \theta_2} \end{pmatrix}$$

In [ ]:
def calcular_jacobiana(theta1, theta2, delta=1e-6):
    """
    Calcula la matriz Jacobiana numéricamente.
    
    Parámetros:
    -----------
    theta1, theta2 : float
        Ángulos articulares [rad]
    delta : float
        Perturbación para derivada numérica
    
    Retorna:
    --------
    np.ndarray : Matriz Jacobiana 2x2
    """
    
    # Posición actual
    x0, y0, _, _, _, _ = cinemática_directa(theta1, theta2)
    
    # Derivadas respecto a theta1
    x1, y1, _, _, _, _ = cinemática_directa(theta1 + delta, theta2)
    dx_dth1 = (x1 - x0) / delta
    dy_dth1 = (y1 - y0) / delta
    
    # Derivadas respecto a theta2
    x2, y2, _, _, _, _ = cinemática_directa(theta1, theta2 + delta)
    dx_dth2 = (x2 - x0) / delta
    dy_dth2 = (y2 - y0) / delta
    
    J = np.array([[dx_dth1, dx_dth2],
                 [dy_dth1, dy_dth2]])
    
    return J

def calcular_dexteridad(theta1, theta2):
    """
    Calcula el índice de dexteridad (número de condición de la Jacobiana).
    
    Parámetros:
    -----------
    theta1, theta2 : float
        Ángulos articulares [rad]
    
    Retorna:
    --------
    float : Número de condición (condicionamiento de la matriz)
    """
    
    J = calcular_jacobiana(theta1, theta2)
    
    # Número de condición
    cond_number = np.linalg.cond(J)
    
    # Determinante
    det_J = np.linalg.det(J)
    
    return cond_number, det_J, J

# Calcular jacobiana en varias posiciones
print("\nAnálisis de Velocidades (Matriz Jacobiana)")
print("="*60)

test_angles = [
    (np.radians(45), np.radians(90)),
    (np.radians(90), np.radians(90)),
    (np.radians(135), np.radians(90))
]

for theta1, theta2 in test_angles:
    cond, det, J = calcular_dexteridad(theta1, theta2)
    print(f"\nθ₁ = {np.degrees(theta1):6.1f}°, θ₂ = {np.degrees(theta2):6.1f}°")
    print(f"  Matriz Jacobiana:\n{J}")
    print(f"  Determinante: {det:.4f}")
    print(f"  Número de Condición: {cond:.4f}")
    if abs(det) < 1e-3:
        print(f"  ⚠ SINGULARIDAD CERCANA")

### Mapa de Singularidades en el Espacio de Trabajo

In [ ]:
def mapear_dexteridad(rango_theta1, rango_theta2, num_puntos=30):
    """
    Crea un mapa de dexteridad del espacio de trabajo.
    """
    
    theta1_range = np.linspace(rango_theta1[0], rango_theta1[1], num_puntos)
    theta2_range = np.linspace(rango_theta2[0], rango_theta2[1], num_puntos)
    
    det_map = np.zeros((num_puntos, num_puntos))
    x_map = np.zeros((num_puntos, num_puntos))
    y_map = np.zeros((num_puntos, num_puntos))
    
    for i, theta1 in enumerate(theta1_range):
        for j, theta2 in enumerate(theta2_range):
            cond, det, _ = calcular_dexteridad(theta1, theta2)
            det_map[j, i] = det
            x_p, y_p, _, _, _, _ = cinemática_directa(theta1, theta2)
            x_map[j, i] = x_p
            y_map[j, i] = y_p
    
    return x_map, y_map, det_map, theta1_range, theta2_range

# Calcular mapas
x_map, y_map, det_map, th1_range, th2_range = mapear_dexteridad(
    (np.radians(0), np.radians(180)), 
    (np.radians(0), np.radians(180)), 
    num_puntos=25
)

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfica 1: Determinante en espacio articular
im1 = axes[0].contourf(np.degrees(th1_range), np.degrees(th2_range), det_map, levels=20, cmap='RdBu_r')
contour = axes[0].contour(np.degrees(th1_range), np.degrees(th2_range), det_map, levels=[0], colors='black', linewidths=2)
axes[0].clabel(contour, inline=True, fontsize=10)
axes[0].set_xlabel('θ₁ [°]', fontsize=11)
axes[0].set_ylabel('θ₂ [°]', fontsize=11)
axes[0].set_title('Determinante Jacobiana (Espacio Articular)')
cbar1 = plt.colorbar(im1, ax=axes[0])
cbar1.set_label('det(J)', fontsize=10)

# Gráfica 2: Determinante en espacio cartesiano
im2 = axes[1].scatter(x_map, y_map, c=det_map, s=50, cmap='RdBu_r', alpha=0.7)
axes[1].set_xlabel('x [mm]', fontsize=11)
axes[1].set_ylabel('y [mm]', fontsize=11)
axes[1].set_title('Determinante Jacobiana (Espacio Cartesiano)')
axes[1].set_aspect('equal')
cbar2 = plt.colorbar(im2, ax=axes[1])
cbar2.set_label('det(J)', fontsize=10)

plt.tight_layout()
plt.show()

print("\nMapa de dexteridad completado.")

## Conclusiones y Observaciones

Este cuaderno proporciona un análisis completo de la cinemática de un manipulador paralelo plano de 2 grados de libertad. Los temas tratados incluyen:

1. **Cinemática Directa**: Cálculo de la posición del efector final dada los ángulos articulares.
2. **Espacio de Trabajo**: Visualización completa de las posiciones alcanzables.
3. **Análisis de Posición**: Gráficas que muestran la relación entre ángulos y posiciones.
4. **Configuraciones Múltiples**: Visualización de diferentes configuraciones del mecanismo.
5. **Análisis de Velocidades**: Cálculo de la matriz Jacobiana y análisis de singularidades.
6. **Mapas de Dexteridad**: Identificación de regiones singulares en el espacio de trabajo.

## Modificaciones Sugeridas

Los parámetros pueden modificarse en la celda de "Parámetros del Mecanismo" para explorar diferentes diseños:
- Cambiar longitudes de eslabones (L1, L2, L3)
- Ajustar la separación de actuadores (d)
- Modificar las condiciones iniciales de ángulos

## Referencias

- Craig, J. J. (2005). Introduction to Robotics: Mechanics and Control
- Tsai, L. W. (1999). Robot Analysis: The Mechanics of Serial and Parallel Manipulators
- Murray, R. M., Sastry, S. S., & Zexiang, L. (1994). A Mathematical Introduction to Robotic Manipulation